# Cryptography-03 — Le chiffrement homomorphe et les preuves à divulgation nulle (C# / .NET)

**Série Cryptography** · Carnet 03 — *sous-série du hub « Information & calcul » (#27092)*

*La série ouvre avec le **carnet 01** (PR #27084) ; le carnet 02 (PR #27127) et le carnet 04 (PR #27369) la poursuivent.*

| Précédent | Suivant | Index |
|---|---|---|
| Cryptography-02 — Du canal bruité au canal adversarial *(PR #27127)* | Cryptography-04 — Le post-quantique *(PR #27369)* | README de la sous-série *(PR #27369)* |

**Toute la crypto « avancée » du carnet tient en deux questions.** La première :
peut-on **calculer sur des données chiffrées** — multiplier, combiner — sans jamais
les déchiffrer, et sans jamais montrer la clé à qui tient le chiffré ? La
seconde : peut-on **prouver qu'on sait un secret** sans que le protocole ne
révèle **rien** de ce secret ? Les deux réponses sont *oui, et on peut
l'exécuter* — avec les primitives **BCL** de .NET, en from-scratch, sur des
schémas jouets dont chaque propriété se **vérifie à l'exécution** (pas en
parole) :

- le **chiffrement partiellement homomorphe** (le RSA jouet du carnet : le
  produit de deux chiffrés se déchiffre en le **produit** des deux messages) ;
- la **preuve à divulgation nulle** (le protocole d'identification de Schnorr :
  complétude, sonorité, et le **témoin négatif** — un preneur « cassé » qui
  réutilise son aléa est **attrapé** par l'extracteur).

## Enjeu — du message caché au message *calculable*, au savoir *prouvé*

Jusqu'ici, chiffrer, c'était **masquer** : le destinataire déchiffre, tout le
reste est exclu. L'homomorphisme change la règle : le chiffré devient un
**objet de calcul** — on combine des chiffrés, le déchiffrement applique la
combinaison, et l'intermédiaire n'apprend rien. La preuve à divulgation nulle
change l'autre règle : on ne montre plus le secret, on **prouve qu'on le
connaît**, et l'échange complet reste *statistiquement indistinguable* d'un
échange avec un simulé qui ne le connaît pas.

## Ponts — les mêmes primitives, du côté de la chaîne

La série **Smart-Contracts** enseigne les mêmes briques depuis l'angle d'un
protocole **déployé** : là, la preuve doit convaincre un tiers sans rien
révéler du secret, et le chiffrement doit protéger un calcul **public**.

| Ici, en BCL .NET, from-scratch | Là-bas, sur chaîne |
|---|---|
| Schnorr, avec son témoin négatif (l'aléa réutilisé est attrapé) | [SC-15 — Zero-Knowledge Proofs](../../SymbolicAI/SmartContracts/04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs-Python.ipynb) : la preuve appliquée au vote vérifiable |
| RSA jouet, homomorphisme multiplicatif démontré par rejeu | [SC-16 — Homomorphic Encryption](../../SymbolicAI/SmartContracts/04-Privacy-Cryptography/SC-16-Homomorphic-Encryption-Python.ipynb) : le calcul chiffré sur une blockchain |
| les deux primitives isolées, prouvées par leurs propriétés | [SmartContracts — index de la série](../../SymbolicAI/SmartContracts/README.md) |

## 1. Le chiffrement homomorphe — une structure qui *survit* au chiffrage

Un chiffrement `E` est **partiellement homomorphe** quand une opération sur les
**chiffrés** correspond à la **même opération** sur les **messages** une fois
déchiffré : ici, la **multiplication**.
`D(E(m1) * E(m2) mod n) = m1 * m2 mod n` : l'intermédiaire peut multiplier des
chiffrés, mais le produit ne se déchiffre **que** par le titulaire de la clé
privée. C'est la propriété exacte que nous allons **exhiber** — sur le RSA jouet
du carnet (petits premiers, arithmétique `long` exacte), pas la promettre.

In [1]:
using System.Security.Cryptography;

// Multiplication modulaire SANS overflow : p vaut 61 bits, a*b depasse long.
// Addition-doubling (algo « russe ») : O(log b), chaque etape reste < m.
static long MulMod(long a, long b, long m)
{
    a = ((a % m) + m) % m;
    b = ((b % m) + m) % m;
    long r = 0;
    while (b > 0)
    {
        if (b % 2 == 1) r = (r + a) % m;
        a = (a + a) % m;
        b /= 2;
    }
    return r;
}

// Exponentiation modulaire par squaring, via MulMod (donc overflow-safe).
static long PowMod(long bas, long exp, long mod)
{
    long result = 1 % mod;
    long b = ((bas % mod) + mod) % mod;
    while (exp > 0)
    {
        if (exp % 2 == 1) result = MulMod(result, b, mod);
        b = MulMod(b, b, mod);
        exp /= 2;
    }
    return result;
}

// Engagement Pedersen : t = SHA-256 de l'element du groupe (big-endian).
static string Commit(long groupElem)
{
    byte[] be = BitConverter.GetBytes(groupElem);
    if (BitConverter.IsLittleEndian) Array.Reverse(be);
    return Convert.ToHexString(SHA256.HashData(be)).ToLowerInvariant();
}

class RsaToy
{
    public long P, Q, N, E, D;
    public RsaToy(long p, long q, long e)
    {
        P = p; Q = q; N = p * q; E = e;
        long phi = (p - 1) * (q - 1);
        for (long d = 1; d < phi; d++)          // e*d = 1 (mod phi)
            if ((e * d) % phi == 1) { D = d; break; }
    }
    public long Encrypt(long m) => PowMod(m, E, N);
    public long Decrypt(long c) => PowMod(c, D, N);
}

// ---- Le RSA jouet : n = 11*13 = 143, e = 7, d = 103 ----
var rsa = new RsaToy(11, 13, 7);
Console.WriteLine($"n = {rsa.N}   phi = {(rsa.P - 1) * (rsa.Q - 1)}   e = {rsa.E}   d = {rsa.D}");

long m = 7;
long c = rsa.Encrypt(m);
Console.WriteLine($"E({m}) = {c}   D({c}) = {rsa.Decrypt(c)}   (le message reapparaît)");

// ---- Le groupe de Schnorr : p premier a 61 bits, g racine primitive ----
long p = 1000000000000000031L;
long g = 13;
long x = 12345;
long y = PowMod(g, x, p);
Console.WriteLine();
Console.WriteLine($"groupe : p = {p}   g = {g} (racine primitive de Z/pZ)");
Console.WriteLine($"secret x = {x}   ->   cle publique y = g^x mod p = {y}");

// ---- Un engagement Schnorr a la main : r, h = g^r, t = H(h) ----
long r = 777;
long h = PowMod(g, r, p);
string t = Commit(h);
Console.WriteLine($"r = {r}   ->   g^r mod p = {h}");
Console.WriteLine($"engagement t = {t[..16]}...   ({t.Length} hex = 32 octets)");

n = 143   phi = 120   e = 7   d = 103


E(7) = 6   D(6) = 7   (le message reapparaît)


groupe : p = 1000000000000000031   g = 13 (racine primitive de Z/pZ)


secret x = 12345   ->   cle publique y = g^x mod p = 336314290501056191


r = 777   ->   g^r mod p = 966808300621636444


engagement t = 24f7258ccb66220e...   (64 hex = 32 octets)


In [2]:
// L'HOMOMORPHISME, exhibe — m1 = 3, m2 = 5 :
//   D( E(m1) * E(m2) mod n )  ==  m1 * m2 mod n
long m1 = 3, m2 = 5;
long c1 = rsa.Encrypt(m1);
long c2 = rsa.Encrypt(m2);
long cProd = (c1 * c2) % rsa.N;          // l'intermediaire multiplie des CHIFFRES
long mProd = rsa.Decrypt(cProd);         // le titulaire de la cle dechiffre le PRODUIT

Console.WriteLine($"E({m1}) = {c1}   E({m2}) = {c2}");
Console.WriteLine($"c1 * c2 mod n        = {cProd}");
Console.WriteLine($"D(c1*c2)             = {mProd}");
Console.WriteLine($"m1 * m2              = {m1 * m2}");
Console.WriteLine(mProd == (m1 * m2) % rsa.N
    ? "OK : le produit des chiffrés se déchiffre en le PRODUIT des messages."
    : "ECHEC : l'homomorphisme ne tient pas.");

// TEMOIN NEGATIF — la meme operation sur les chiffrés, mais l'ADDITION :
long cSomme = (c1 + c2) % rsa.N;
long mSomme = rsa.Decrypt(cSomme);
Console.WriteLine();
Console.WriteLine($"D(c1 + c2) = {mSomme}   alors que m1 + m2 = {m1 + m2}");
Console.WriteLine(mSomme == (m1 + m2) % rsa.N
    ? "inattendu : l'addition passerait aussi."
    : "OK : l'addition NE PASSE PAS — le schema est MULTIPLICATIF, et c'est precisement ce qui se prouve.");

E(3) = 42   E(5) = 47


c1 * c2 mod n        = 115


D(c1*c2)             = 15


m1 * m2              = 15


OK : le produit des chiffrés se déchiffre en le PRODUIT des messages.


D(c1 + c2) = 67   alors que m1 + m2 = 8


OK : l'addition NE PASSE PAS — le schema est MULTIPLICATIF, et c'est precisement ce qui se prouve.


### Lecture du résultat

`E(3) = 42`, `E(5) = 47`. L'intermédiaire tient les deux chiffrés et leur produit modulo `n` : `42 * 47 mod 143 = 115` — un chiffre qu'**aucun** des deux ne peut déchiffrer. Le titulaire de la clé obtient `D(115) = 15 = 3 * 5` : la combinaison s'est faite **sur les chiffrés**, le résultat n'existe en clair **qu'au moment du déchiffrement**. Le témoin négatif ferme la parenthèse : `D(c1 + c2) = 67` alors que `m1 + m2 = 8` — l'addition **ne passe pas**. Le schéma n'est donc pas « homomorphe pour tout », il est homomorphe **pour la multiplication, et pour elle seule** : c'est la propriété exacte qu'on a mesurée, pas une assertion.

*(Le RSA réel des systèmes est multiplicatif pour la même raison algébrique — `m^e mod n` commute avec la multiplication — mais ses premiers sont de 2048 bits : on ne peut plus raisonner en `long`. Le jouet garde la structure, pas la sécurité ; c'est le rôle qu'il joue ici.)*

## 2. La preuve à divulgation nulle — prouver *que* sans révéler *quoi*

Deuxième question du carnet : **comment convaincre que je connais un secret, sans révéler le secret ?**

Le protocole de **Schnorr** (1991) est la réponse la plus courte. Le prouveur tient un secret `x` (un exposant) dont la trace publique est `y = g^x mod p` dans un groupe cyclique de premier `p`. La danse a trois temps, et l'ordre en est la clé :

1. **S'engager** : le prouveur tire `r` au hasard, publie `t = H(g^r mod p)` — un engagement qui prouve qu'il *a* un `r`, sans dire lequel ;
2. **Le défi** : le vérificateur répond par un aléa `c` — **après** l'engagement, donc il ne peut pas le fabriquer sur mesure ;
3. **Répondre** : le prouveur publie `s = r + c·x mod (p−1)`. Le vérificateur contrôle `g^s == h · y^c mod p`, où `h = g^r`. Si l'égalité tient, il est convaincu que le prouveur connaissait `x` *au moment de l'engagement*.

Les trois propriétés d'un ZK sont vérifiées **par exécution** — pas par lecture :

- **Complétude** : un prouveur honnête qui tient `x` est accepté (5/5 exécutions) ;
- **Sonorité** : un tricheur sans `x` ne peut pas imiter *à la demande* — et le carnet exhibe le cas où c'est possible : **réutiliser le même `r`** laisse le secret s'échapper, le carnet le récupère ;
- **Divulgation nulle** : on *simule* une transcription valide **sans connaître `x`** — si la transcription ne porte pas plus d'information que ce qu'un simulateur peut fabriquer, elle n'en porte aucune.

Le groupe est le même que pour les exercices du carnet 02 : premier `p` de 61 bits, `g` racine primitive. Tout le code est du BCL — `System.Security.Cryptography.SHA256` pour l'engagement, et `PowMod` de la section 1 pour l'arithmétique modulaire.

In [3]:
record SchnorrTranscript(string T, long C, long S)
{
    // T est un hash hexa (64 car.) en usage reel ; on tronque a 12 car.
    // (garde de longueur : un placeholder plus court s'affiche tel quel).
    public override string ToString() => $"t={(T.Length >= 12 ? T[..12] : T)}...  c={C}  s={S}";
}

class SchnorrProver
{
    public long X, P, G;
    public SchnorrProver(long x, long p, long g) { X = x; P = p; G = g; }
    public long Y => PowMod(G, X, P);

    // 1. S'engager : r aléatoire, h = g^r, t = H(h)
    //    (nommé Engage, pas Commit, pour ne pas masquer le helper Commit(h))
    public (long R, string T, long H) Engage()
    {
        long r = RandomNumberGenerator.GetInt32(int.MaxValue);
        long h = PowMod(G, r, P);
        return (r, Commit(h), h);
    }

    // 3. Répondre : s = r + c*x mod (p-1)
    public long Respond(long r, long c) => (r + c * X) % (P - 1);
}

// 2. Vérifier : g^s  ==  h * y^c  mod p   (le verificateur ne connait que y)
static bool Verify(SchnorrTranscript tr, long h, long y, long p, long g)
{
    long gauche = PowMod(g, tr.S, p);
    long droite = MulMod(h, PowMod(y, tr.C, p), p);
    return gauche == droite;
}

var prover = new SchnorrProver(x, p, g);
Console.WriteLine($"prouveur : y = g^x mod p = {prover.Y}");
Console.WriteLine(prover.Y == y ? "cohérent avec la section 1." : "incohérent !");

// COMPLETUDE : 5 epreuves honnetes, le verificateur accepte 5/5
Console.WriteLine();
int ok = 0;
var rng = new Random(20261118);
for (int i = 0; i < 5; i++)
{
    var (r, t, h) = prover.Engage();
    long c = 1 + rng.Next(999);
    long s = prover.Respond(r, c);
    var tr = new SchnorrTranscript(t, c, s);
    bool acc = Verify(tr, h, y, p, g);
    ok += acc ? 1 : 0;
    Console.WriteLine($"epreuve {i + 1} : {tr}   ->   {(acc ? "accepte" : "refuse")}");
}
Console.WriteLine($"\nCOMPLETUDE : {ok}/5 epreuves honnetes acceptees.");

prouveur : y = g^x mod p = 336314290501056191


cohérent avec la section 1.


epreuve 1 : t=c5c6f473270d...  c=663  s=854956840   ->   accepte


epreuve 2 : t=7ef9d6c7d82f...  c=52  s=929503637   ->   accepte


epreuve 3 : t=87ce26c0bbd0...  c=923  s=858392827   ->   accepte


epreuve 4 : t=6ea2d4f94d87...  c=961  s=1020304013   ->   accepte


epreuve 5 : t=56401ebeac37...  c=55  s=1184151104   ->   accepte



COMPLETUDE : 5/5 epreuves honnetes acceptees.


### Sonorité — par le cas où elle ne tient PAS

Le verbe « prouver » exige un témoin négatif : **que se passe-t-il quand le prouveur se trompe ?** Le piège classique est de **réutiliser le même `r`** pour deux épreuves (même engagement `t`, deux défis `c1`, `c2`). Un prouveur honnête tire `r` à chaque fois — réutiliser `r` est une faute de protocole, pas une attaque. Mais si le fait, un observateur qui lit les deux transcriptions **publiques** `(t, c1, s1)` et `(t, c2, s2)` peut en extraire le secret : les deux équations `s = r + c·x` se soustraient, `r` tombe, et il ne reste qu'une inconnue. Le carnet ne décrit pas la fuite, il **l'exécute** : deux transcriptions valides, puis la récupération de `x` par l'extracteur.

In [4]:
// Inverse modulo (p-1) par l'algorithme euclidien etendu.
static long ModInverse(long a, long m)
{
    a = ((a % m) + m) % m;
    long oldR = a, r = m;
    long oldS = 1, s = 0;
    while (r != 0)
    {
        long q = oldR / r;
        (oldR, r) = (r, oldR - q * r);
        (oldS, s) = (s, oldS - q * s);
    }
    return ((oldS % m) + m) % m;
}

// Le "prouveur" FAUTE : meme r pour les deux epreuves (meme engagement t).
long rFixe = 42;
long hFixe = PowMod(g, rFixe, p);
string tFixe = Commit(hFixe);
Console.WriteLine($"r reutilise = {rFixe}   ->   t = {tFixe[..16]}...");

// Deux defis de PARITE DISTINCTE : c1 - c2 = 3 (impair) -> inversible modulo (p-1).
long c1 = 11, c2 = 8;
long s1 = (rFixe + c1 * x) % (p - 1);
long s2 = (rFixe + c2 * x) % (p - 1);
var tr1 = new SchnorrTranscript(tFixe, c1, s1);
var tr2 = new SchnorrTranscript(tFixe, c2, s2);

Console.WriteLine($"epreuve 1 : {tr1}   ->   {(Verify(tr1, hFixe, y, p, g) ? "accepte" : "refuse")}");
Console.WriteLine($"epreuve 2 : {tr2}   ->   {(Verify(tr2, hFixe, y, p, g) ? "accepte" : "refuse")}");
Console.WriteLine("les deux transcriptions passent la verification — et elles sont PUBLIQUES.");

// L'EXTRACTEUR : s = r + c*x  ->  (s1 - s2) = (c1 - c2)*x  mod (p-1)
long dc = c1 - c2;
long invDc = ModInverse(dc, p - 1);
long xRecupere = MulMod(s1 - s2, invDc, p - 1);

Console.WriteLine();
Console.WriteLine($"(s1 - s2) = {(s1 - s2)}   (c1 - c2) = {dc}");
Console.WriteLine($"x recupere = {xRecupere}   (secret reel : {x})");
Console.WriteLine(xRecupere == x
    ? "ATTRAPE : reutiliser r livre x — les deux transcriptions publiques suffisent."
    : "ECHEC : l'extracteur ne retrouve pas x.");

r reutilise = 42   ->   t = 77916c09fd729a2e...


epreuve 1 : t=77916c09fd72...  c=11  s=135837   ->   accepte


epreuve 2 : t=77916c09fd72...  c=8  s=98802   ->   accepte


les deux transcriptions passent la verification — et elles sont PUBLIQUES.


(s1 - s2) = 37035   (c1 - c2) = 3


x recupere = 12345   (secret reel : 12345)


ATTRAPE : reutiliser r livre x — les deux transcriptions publiques suffisent.


### Divulgation nulle — par la *simulation*

La propriété la plus subtile est la **divulgation nulle** : la transcription ne doit rien révéler de `x`. Le test standard est un **simulateur** : un algorithme qui **ne connaît pas `x`** doit pouvoir fabriquer des transcriptions *statistiquement indiscernables* des vraies. S'il y parvient, la transcription ne porte manifestement pas d'information sur `x` — sinon le simulateur, qui n'a pas `x`, ne pourrait pas la produire.

La marche arrière est courte : au lieu de partir de `r` et de dériver `t` puis `s`, on **choisit `s` et `c` au hasard**, et on définit `t` en arrière : `t = H(g^s · y^(-c) mod p)`. Le vérificateur, qui ne fait que contrôler `g^s == h·y^c`, ne distingue pas une transcription simulée d'une vraie — et le simulateur n'a jamais touché à `x`.

In [5]:
// Le SIMULATEUR : choisit s et c, derivo h en arriere — SANS x.
var rngZk = new Random(99);
Console.WriteLine();
int vok = 0;
for (int i = 0; i < 3; i++)
{
    long s = rngZk.Next(int.MaxValue);
    long c = 1 + rngZk.Next(999);
    // h = g^s * y^(-c)  =  g^s * y^(p-1-c)   (Fermat)  — seulement y, pas x
    long hSim = MulMod(PowMod(g, s, p), PowMod(y, (p - 1 - c) % (p - 1), p), p);
    string tSim = Commit(hSim);
    var tr = new SchnorrTranscript(tSim, c, s);
    bool acc = Verify(tr, hSim, y, p, g);
    vok += acc ? 1 : 0;
    Console.WriteLine($"simulation {i + 1} : {tr}   ->   {(acc ? "valide" : "invalide")}   (sans x)");
}
Console.WriteLine($"\nDIVULGATION NULLE : {vok}/3 transcriptions valides fabriquees SANS le secret x.");

simulation 1 : t=8f645cab0d06...  c=865  s=958527983   ->   valide   (sans x)


simulation 2 : t=a0b59df8a9c3...  c=689  s=2078407144   ->   valide   (sans x)


simulation 3 : t=174ac683f4b3...  c=75  s=1939438418   ->   valide   (sans x)



DIVULGATION NULLE : 3/3 transcriptions valides fabriquees SANS le secret x.


### Lecture du résultat

Les trois propriétés d'une preuve à divulgation nulle sont maintenant **exécutées**, pas lues :

- **Complétude** — `COMPLETUDE : 5/5` : un prouveur honnête qui tient `x` est accepté à chaque épreuve. Le protocole ne rejette jamais un prouveur légitime.
- **Sonorité** — les deux transcriptions à `r` réutilisé **passent** la vérification, et l'extracteur en tire `x = 12345` (le secret exact) par une simple soustraction : `(s1 − s2) = (c1 − c2)·x`. Le témoin négatif montre *pourquoi* le protocole honnête est sûr : il tire `r` à chaque fois, donc un écouteur ne possède jamais deux équations pour une même inconnue.
- **Divulgation nulle** — `3/3` transcriptions **valides** produites par un simulateur qui **ne connaît pas `x`**. Si la transcription ne porte pas plus d'information que ce qu'un simulateur aveugle peut fabriquer, elle n'en porte aucune sur le secret.

*(Limites du jouet : `p` à 61 bits, défis `c` non hachés et de petite taille, engagement `H(g^r)` et non `H(g^r ‖ r)`. Les protocoles de production — Schnorr et ses descendants en signature — durcissent chacun de ces points : hachage du défi, élargissement du domaine, premiers de 256 bits ou plus. La structure, elle, ne change pas : **engagement, défi, réponse**.)*

## Synthèse

| Idée | Comment le carnet l'exécute |
|---|---|
| **Chiffrement homomorphe** | RSA jouet : `D(E(m1)·E(m2)) = m1·m2`, exhibé (`15`), et **borné** par un témoin négatif (l'addition ne passe pas). |
| **Engagement** | `t = H(g^r)` : le hachage scelle l'aléa, le défi arrive *après* — c'est ce qui rend la sonorité possible. |
| **Preuve ZK (Schnorr)** | Complétude `5/5`, sonorité par l'extracteur qui récupère `x`, divulgation nulle par un simulateur **sans** `x`. |

Le fil conducteur des deux moitiés est le même : **une structure algébrique** (la commutation du chiffrage, l'équation `g^s = h·y^c`) dont on **exécute** les propriétés — y compris les cas où elle ne tient pas. Le chiffrement homomorphe combine une opération *sur les chiffrés* ; la preuve ZK combine un engagement, un défi et une réponse. Et dans les deux cas, le **témoin négatif** (l'addition qui échoue, le `r` réutilisé qui fuit) est ce qui donne au positif sa valeur : sans lui, « ça passe 5/5 » ne prouverait rien.

**Renvois** — le **carnet 01** de la série (PR #27084) l'ouvre ; le carnet 02 de la série (*Du canal bruité au canal adversarial*, PR #27127) pose les groupes et le RSA que ce carnet manipule ; le **carnet 04** de la série traitera le post-quantique (les primitives qui survivent à un ordinateur quantique). Côté mise en œuvre, la série **Smart-Contracts** branche ces primitives sur chaîne : [SC-15 — Zero-Knowledge Proofs](../../SymbolicAI/SmartContracts/04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs-Python.ipynb) et [SC-16 — Homomorphic Encryption](../../SymbolicAI/SmartContracts/04-Privacy-Cryptography/SC-16-Homomorphic-Encryption-Python.ipynb) ; côté coût, la série **Complexity** pose ce que « prouver » coûte en calcul.

## Exercices

Trois exercices, un par propriété. Chaque fonction est un stub `// TODO etudiant` — le carnet s'exécute de bout en bout, exercices non complétés compris. Complétez-les, puis relancez la cellule pour vérifier votre réponse contre les valeurs que le carnet a affichées.

In [6]:
// EXERCICE 1 — homomorphisme : completez MultiplierChiffres.
// L'intermediaire tient les deux chiffrés et les multiplie ; le titulaire de
// la cle dechiffre. Le resultat doit etre le PRODUIT des messages.
static long MultiplierChiffres(long a, long b)
{
    // TODO etudiant : chiffrer a et b (rsa.Encrypt), multiplier les chiffrés
    // modulo rsa.N, dechiffrer (rsa.Decrypt), retourner le resultat.
    return 0;
}

// Test : a = 3, b = 5 — le carnet a exhibe que le resultat attendu est 15.
Console.WriteLine($"exo 1 : MultiplierChiffres(3, 5) = {MultiplierChiffres(3, 5)}   (attendu : 15)");

exo 1 : MultiplierChiffres(3, 5) = 0   (attendu : 15)


In [7]:
// EXERCICE 2 — sonorité : completez ExtracterSecret.
// Deux transcriptions a r REUTILISE : (s1, c1) et (s2, c2).
// s = r + c*x  (mod p-1)  ->  (s1 - s2) = (c1 - c2)*x  (mod p-1).
// Retrouvez x.
static long ExtracterSecret(long s1, long s2, long c1, long c2, long p)
{
    // TODO etudiant : (s1 - s2) * (c1 - c2)^{-1}  modulo (p - 1).
    // ModInverse et MulMod sont deja definis dans le carnet — reusez-les.
    return 0;
}

// Test : les deux transcriptions de la section Sonorite (c1 = 11, c2 = 8).
long rT = 42;
long s1T = (rT + 11 * x) % (p - 1);
long s2T = (rT + 8  * x) % (p - 1);
Console.WriteLine($"exo 2 : ExtracterSecret(...) = {ExtracterSecret(s1T, s2T, 11, 8, p)}   (attendu : {x})");

exo 2 : ExtracterSecret(...) = 0   (attendu : 12345)


In [8]:
// EXERCICE 3 — divulgation nulle : completez SimulerTranscript.
// Fabriquez une transcription VRAIE (qui passe Verify) SANS connaitre x :
// choisissez s et c au hasard, puis h = g^s * y^(p-1-c)  mod p.
static SchnorrTranscript SimulerTranscript(long y, long p, long g)
{
    // TODO etudiant : tirer s et c au hasard, deriver h = g^s * y^(p-1-c) mod p
    // SANS x, puis renvoyer new SchnorrTranscript(Commit(h), c, s).
    return new SchnorrTranscript("0", 0, 0);
}

// Test : le stub renvoie une transcription nulle (placeholder).
var trSim = SimulerTranscript(y, p, g);
Console.WriteLine($"exo 3 : SimulerTranscript -> {trSim}   (placeholder du stub)");

exo 3 : SimulerTranscript -> t=0...  c=0  s=0   (placeholder du stub)


## À retenir

- **Homomorphe partiel** : une opération sur les chiffrés commute avec le
  déchiffrement — ici la **multiplication** du RSA, exhibée et bornée par un
  témoin négatif (l'addition ne passe pas).
- **Engagement** : `t = H(g^r)` — le hachage **scelle** l'aléa ; c'est ce qui
  force le défi à venir *après*, et donc la sonorité.
- **Schnorr** : complétude (l'honnête passe toujours), sonorité (la réutilisation
  de `r` livre `x` à un simple écouteur), divulgation nulle (un simulé **sans**
  `x` reproduit l'échange).
- **Le jouet est un télescope, pas une arme** : 61 bits, défis non hachés — la
  structure est celle des protocoles réels, la sécurité n'en est pas.

## Bibliographie

- J. von O. Schnorr, *Efficient Identifications and Protections against
  Impersonation and Substitution*, CRYPTO'89 (1990) — le protocole d'identification.
- D. Boneh, V. Shoup, *A Graduate Course in Applied Cryptography* (gratuit,
  boneh-shoup) — chapitres sur les preuves ZK et le RSA.
- J. Katz, M. Lindell, *Introduction to Modern Cryptography*, 3e éd., CRC — le
  cadre des définitions (complétude / sonorité / ZK) utilisées ici.